
# Neural network for Collaborative Filtering (NCF), 2017


models
- gmf
- mlp: 32 -> 16 -> 8
- neumf: gmf + mlp

exp setting
- activation: relu
- loss: log loss
- mlp init: gaussian(0, 0.01), adam
- neulml init: mlp, gmf pre-trained, sgd
- metrics: HR@10, NDCG@10
- model selection: leave-one-out
- negative sampling 

In [1]:
import os, json, bson

import numpy as np
import pandas as pd
from tqdm import tqdm

import tensorflow as tf

2023-07-20 07:54:39.157717: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-20 07:54:39.345953: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-07-20 07:54:40.037095: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64
2023-07-20 07:54:40.037189: W tensorflow/stream_

# Data Load 

In [2]:
pinterest_dir = '../data/pinterest_iccv/'

In [3]:
f = open(os.path.join(pinterest_dir, 'subset_iccv_board_pins.bson'), 'rb')
board_pins = []
for rows in bson.decode_file_iter(f):
    board_pins.append(rows)
len(board_pins)

46000

# Prepare Train/Test data

- leave-one-out evaluation (positive)
    - 테스트 데이터: 유저의 latest 데이터
    - 학습 데이터: 유저의 나머지 데이터
- 학습 시에는 모든 negative 데이터를 포함
- 평가 시에는 positive test data 하나와 나머지 샘플링한 unobserved 100개 데이터를 합하여 랭킹
- 위 랭킹 결과의 top k에 대해서 positive 가 포함되어 있으면 hit, 그 위치가 어디인지를 NDCG가 측정

In [4]:
all_pins = set()

for b in board_pins:
    for p in b['pins']:
        all_pins.add(p)

all_pins = list(all_pins)
len(all_pins)

2565241

In [5]:
user_set = set([b['board_id'] for b in board_pins])

In [6]:
pins = [len(b['pins']) for b in board_pins]
num_negative = int(sum(pins)/len(pins))
num_negative

55

In [7]:
pin_map = {p:i for i,p in enumerate(all_pins)}
user_map = {u:i for i,u in enumerate(user_set)}

len(pin_map.keys()), len(user_map.keys())

(2565241, 46000)

In [8]:
train = [] # user, item, score
test = [] # user, item, score

for b in tqdm(board_pins):
    train.extend([[user_map[b['board_id']], pin_map[p], 1] for p in b['pins'][:-1]])
    test.append([user_map[b['board_id']], pin_map[b['pins'][-1]], 1])

100%|██████████| 46000/46000 [00:06<00:00, 7162.25it/s] 


In [9]:
def sampling(num_sample, all_pins, without, user_board):
    idx = 0
    sampled = []
    while idx < num_sample:
        randint = np.random.randint(0, 5043)
        if all_pins[randint] in without:
            continue
        else:
            sampled.append([user_map[user_board], pin_map[all_pins[randint]], 0])
            idx += 1
    return sampled

for b in tqdm(board_pins):
    train.extend(sampling(num_negative, all_pins, b['pins'], b['board_id']))
    test.extend(sampling(100, all_pins, b['pins'], b['board_id']))

100%|██████████| 46000/46000 [00:36<00:00, 1265.02it/s]


In [10]:
train, test = np.array(train), np.array(test)
train.shape, test.shape

((5049242, 3), (4646000, 3))

In [11]:
np.random.shuffle(train)
np.random.shuffle(test)

샘플링은 유저별로 55번씩하면 되는데 positive에 있는 데이터가 들어가면 안됨

In [12]:
all_users = {tp[0] for tp in np.concatenate([train,test], axis=0)}
all_items = {tp[1] for tp in np.concatenate([train,test], axis=0)}
len(all_users), len(all_items)

(46000, 2565241)

# GMF (generalization matrix factorization)

## build

In [13]:
def get_gmf(num_users, num_items, latent_dim):
    
    user_input = tf.keras.layers.Input(shape=1, dtype="float32")
    item_input = tf.keras.layers.Input(shape=1, dtype="float32")

    user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    user_flatten = tf.keras.layers.Flatten()(user_embedding)
    item_flatten = tf.keras.layers.Flatten()(item_embedding)

    x = tf.keras.layers.Lambda(lambda x: tf.multiply(x[0], x[1]))([user_flatten, item_flatten])
    x = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32", name='last_layer',
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    model = tf.keras.Model([user_input, item_input], x)
    return model

In [14]:
# hyperparams
learning_rate = 0.001


mirrored_strategy = tf.distribute.MirroredStrategy()

with mirrored_strategy.scope():
  gmf_model = get_gmf(len(all_users), len(all_items), 16)

gmf_model.summary()

2023-07-20 07:55:51.301029: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-20 07:55:51.302811: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-20 07:55:51.314126: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-20 07:55:51.315723: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-07-20 07:55:51.317322: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from S

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 user_embedding (Embedding)     (None, 1, 16)        736000      ['input_1[0][0]']                
                                                                                                  
 item_embedding (Embedding)     (None, 1, 16

In [15]:
gmf_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="binary_crossentropy")

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


## training

In [16]:
gmf_model.fit([train[:, 0], train[:, 1]], train[:, 2], epochs=10, batch_size=2048)

Epoch 1/10
INFO:tensorflow:batch_all_reduce: 2 all-reduces with algorithm = nccl, num_packs = 1
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:GPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1').
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:GPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1').
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:batch_all_reduce: 2 all-reduces with algorithm = nccl, num_packs = 1
INFO:tensorflow:Reduce to

## prediction

In [17]:
gmf_test_pred = gmf_model.predict([test[:, 0], test[:, 1]], batch_size=2048)

2269/2269 [==============================] - 8s 3ms/step


## Multi Layer Perceptron

In [ ]:
def get_mlp(num_users, num_items, latent_dim, layer_dims):
    
    user_input = tf.keras.layers.Input(shape=1, dtype="float32")
    item_input = tf.keras.layers.Input(shape=1, dtype="float32")

    user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    user_flatten = tf.keras.layers.Flatten()(user_embedding)
    item_flatten = tf.keras.layers.Flatten()(item_embedding)

    x = tf.keras.layers.Concatenate()([user_flatten, item_flatten])

    # multi layers
    for i, l in enumerate(layer_dims):
        x = tf.keras.layers.Dense(units=l, activation="relu", dtype="float32", name=f'{i}st_layer',
            kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    # last layer
    x = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32", name='last_layer',
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    model = tf.keras.Model([user_input, item_input], x)
    return model

In [ ]:
mlp_model = get_mlp(len(all_users), len(all_items), 16, [32, 16, 8])
mlp_model.summary()

In [ ]:
learning_rate = 0.00001
mlp_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="binary_crossentropy", metrics=['acc'])

In [ ]:
mlp_model.fit([train[:, 0], train[:, 1]], train[:, 2], epochs=10, batch_size=2048)

# NeuMF (Neural Matrix Factorization)

In [ ]:
def get_neuMF(num_users, num_items, latent_dim, layer_dims):

    # common inputs
    user_input = tf.keras.layers.Input(shape=1, dtype="float32")
    item_input = tf.keras.layers.Input(shape=1, dtype="float32")

    # gmf branch
    gmf_user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='gmf_user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    gmf_item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='gmf_item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    gmf_user_flatten = tf.keras.layers.Flatten()(gmf_user_embedding)
    gmf_item_flatten = tf.keras.layers.Flatten()(gmf_item_embedding)

    gmf_multiply = tf.keras.layers.Lambda(lambda x: tf.multiply(x[0], x[1]), name='gmf_multiply')(
        [gmf_user_flatten, gmf_item_flatten])

    # mlp branch
    mlp_user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='mlp_user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    mlp_item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='mlp_item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    mlp_user_flatten = tf.keras.layers.Flatten()(mlp_user_embedding)
    mlp_item_flatten = tf.keras.layers.Flatten()(mlp_item_embedding)

    x = tf.keras.layers.Concatenate()([mlp_user_flatten, mlp_item_flatten])

    # multi layers
    for i, l in enumerate(layer_dims):
        x = tf.keras.layers.Dense(units=l, activation="relu", dtype="float32", name=f'{i}st_layer',
            kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)
        
    # concat branches
    x = tf.keras.layers.Concatenate()([gmf_multiply, x])

    # last layer
    neumf_last_layer = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32", name='last_layer',
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42), bias_constraint=None)(x)

    neumf_model = tf.keras.Model([user_input, item_input], neumf_last_layer)


In [ ]:
neuMF_model = get_neuMF(len(all_users), len(all_items), 16, [32, 16, 8])
neuMF_model.summary()

## load saved weights

In [ ]:
learning_rate = 0.00001
neuMF_model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate), loss="binary_crossentropy", metrics=['acc'])

## Training

In [ ]:
neuMF_model.fit([train[:, 0], train[:, 1]], train[:, 2], epochs=10, batch_size=2048)

# Evaluation

## NDCG

In [18]:
gmf_test_pred

array([[3.2802136e-06],
       [4.4711422e-05],
       [3.6975984e-05],
       ...,
       [5.1606380e-06],
       [1.1082555e-07],
       [1.2884149e-06]], dtype=float32)

In [22]:
import imp
import evaluate
imp.reload(evaluate.ndcg)

<module 'evaluate.ndcg' from '/home/leejuyeon/etc/recsys/models/../evaluate/ndcg.py'>

In [23]:
import sys
sys.path.insert(0, '..')
from evaluate import ndcg #import average_ndcg

topn = 10
ndcg.average_ndcg(test[:, 0], gmf_test_pred, test[:, 2], topn, len(all_users))

0.9981521739130435

In [ ]:
users = test[:, 0]
pred = gmf_test_pred
gt = test[:, 2]
num_users = len(all_users)

# 유저별로 결과를 모아줄 리스트 생성 (gt에는 1,0, pred에는 예측 확률)
gt_list = [[] for x in range(num_users)]
pred_list = [[] for x in range(num_users)]
# binary = [[] for x in range(all_user)]

for u,p,g in zip(users, pred, gt):
    gt_list[u].append(g)
    pred_list[u].append(p[0])

gt_list = np.array(gt_list)
pred_list = np.array(pred_list)

print(pred_list.shape, gt_list.shape)

(46000, 101) (46000, 101)


In [42]:
pred_list

array([[9.9975997e-01, 1.8148397e-05, 1.8600519e-04, ..., 2.2864492e-06,
        8.0455459e-07, 8.9324521e-06],
       [9.9969900e-01, 3.4472036e-05, 2.3118544e-05, ..., 1.9958099e-05,
        1.7586115e-06, 9.9932481e-07],
       [9.9980694e-01, 7.4044269e-06, 2.8772063e-06, ..., 3.9992465e-05,
        1.0271669e-05, 4.1753367e-05],
       ...,
       [9.9969578e-01, 3.9538195e-06, 8.6980992e-07, ..., 1.0778148e-06,
        3.5135376e-06, 2.4728943e-05],
       [9.9987245e-01, 7.6892575e-06, 1.9968458e-05, ..., 5.7336374e-06,
        5.1538559e-06, 4.3896740e-05],
       [9.9975950e-01, 4.2628625e-09, 2.8080481e-06, ..., 2.9827083e-09,
        1.0214349e-05, 5.0363019e-06]], dtype=float32)

In [40]:
pred_sorted_index = np.argsort(pred_list, axis=1)
gt_sorted = np.take_along_axis(gt_list, pred_sorted_index, axis=1)

In [41]:
gt_sorted

array([[0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 1],
       ...,
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 1]])

In [33]:
sum_ndcg = 0

ideal_result = np.array([1] + [0]*(topn-1))
common_idcg = ndcg.discounted_cumulated_gain(ideal_result, topn)
print(common_idcg)

0.1


In [34]:
for g in tqdm(gt_sorted):
    dcg = ndcg.discounted_cumulated_gain(g[::-1], topn)
    sum_ndcg += dcg/common_idcg
sum_ndcg

100%|██████████| 46000/46000 [00:01<00:00, 41544.70it/s]


45897.0

In [37]:
sum_ndcg/num_users

0.9977608695652174

In [28]:
np.argsort(np.array([[1, 2], [4, 3]]), axis=1)

array([[0, 1],
       [1, 0]])

## Hit rate

In [ ]:
# test_pred = gmf_model.predict([test[:, 0], test[:, 1]], batch_size=2048)

In [ ]:
# test_pred_2 = test_pred >= 0.5
# test_pred_2

In [ ]:
# def average_hit_rate(users, gt, pred, topn, num_users):
#     gt_list = [[] for x in range(num_users)]
#     pred_list = [[] for x in range(num_users)]

#     for g,p,u in zip(gt, pred, users):
#         # if len(topk_gt_list[u]) > topn:
#         #     continue
#         # else:
#         gt_list[u].append(p)
#         pred_list[u].append(p)
    
#     hits_dict = 0
#     div = 0
#     for g in gt_list:
#         if len(g) > 0:
#             div += 1
#             g.sort(reverse=True)
#             # print(g)
#             if 1 in g[:10]:
#                 hits_dict += 1
#     # print(hits_dict)
        
#     return hits_dict, div

In [ ]:
topn = 10
hits_dict, div = \
    average_hit_rate(test[:, 0], test_pred_2, test[:, 2], topn, len(all_users))

In [ ]:
hits_dict, div

In [ ]:
# 두 지표 모두 보완 필요

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
import imp
import evaluate
imp.reload(evaluate.ndcg)

In [ ]:
from evaluate import ndcg #import average_ndcg

ndcg.average_ndcg(test[:, 0], test_pred, test[:, 2], topn, len(all_users))

In [ ]:
[np.log2(i+1) for i in range(1, 10+1)]

In [ ]:
hits_dict

## ndcg

In [ ]:
from ndcg import ndcg, discounted_cumulated_gain

In [ ]:
ndcg(test[:, 2], test_pred_2, topn)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
test_pred_2 = test_pred >= 0.5

In [ ]:
accuracy_score(test_y, test_pred_2)

In [ ]:
from sklearn.metrics import ndcg_score

In [ ]:
user_scores = {u:[] for u in all_users}
for u in zip(test_X_user, test_y):
    user_scores[u].append()

In [ ]:
ndcg_score

---

In [ ]:
@tf.function
def hit_ratio(y_true, y_pred):
    y_pred = [int(x >= 0.5) for x in y_pred]
    hits = [int(x==y==1) for x,y in zip(y_pred, y_true)]
    return sum(hits)/sum(y_pred)

In [ ]:
hit_ratio([1, 0, 1], [0.2, 0.1, 0.9])

In [ ]:
import tensorflow as tf
tf.reduce_sum([0,0,1])

In [ ]:
# class hit_ratio(tf.keras.metrics.Metric):
#     def __init__(self, name = 'hit_ratio', **kwargs):
#         super(hit_ratio, self).__init__(**kwargs)
#         self.hits = None

#     def update_state(self, y_true, y_pred, sample_weight=None):
#         y_true = tf.cast(y_true, tf.bool)
#         y_pred = tf.math.greater(y_pred, 0.5)
#         self.hits = tf.logical_and(tf.equal(y_true, y_pred), tf.equal(y_true, 1))
#         self.targets = tf.reduce_sum(y_pred)
        
#     def reset_state(self):
#         self.hits.assign(0)

#     def result(self):
#         return tf.reduced_sum(self.hits)/sum(y_pred)

In [ ]:
# def hr_keras(y_true, y_pred):
#     score = tf.py_function(func=hit_ratio, inp=[y_true, y_pred], Tout=tf.float64,  name='custom_hr') # tf 2.x
#     #score = tf.py_func( lambda y_true, y_pred : mse_AIFrenz(y_true, y_pred) , [y_true, y_pred], 'float32', stateful = False, name = 'custom_mse' ) # tf 1.x
#     return score

In [ ]:
class NCF(tf.keras.Model):

    def __init__(self):
        pass

In [ ]:
# gmf
class GMF(NCF):
    def __call__(self):
        pass

# gmf
class MLP(NCF):
    def __call__(self):
        pass

# gmf
class NeuMF(NCF):
    def __call__(self):
        pass